<a href="https://colab.research.google.com/github/benjamin-holman/Blackjack-dealer-strategy-analysis/blob/main/Blackjack%20Project" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#random card generator

import random
from collections import Counter
from functools import lru_cache
from collections import defaultdict
import pandas as pd

RANKS = ['2', '3', '4', '5', '6', '7', '8', '9', '10', 'jack', 'queen', 'king', 'ace']

VALUES = {
    '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9, '10': 10,
    'jack': 10, 'queen': 10, 'king': 10,
    'ace': 11,
}

#we treat it as an infinite deck, i.e the probability of drawing each number is 1/13 at all times

def draw():
    return random.choice(RANKS)

def hand_value(cards):
    total = sum(VALUES[card] for card in cards)
    aces = cards.count('ace')          # aces at 11
    while total > 21 and aces > 0:
        total -= 10                    # turn an ace into 1
        aces -= 1
    return total

def dealer_hand(threshold=17):
    cards = [draw(), draw()]
    while hand_value(cards) < threshold:
        cards.append(draw())
    return cards, hand_value(cards)

def is_blackjack(cards):
    return hand_value(cards) == 21 and len(cards) == 2

In [15]:
#part 1
#dealer results with dealer having different stop thresholds

def add_card(total, ace_count, card):

    total += card
    if card == 11:
        ace_count += 1
    while total > 21 and ace_count > 0:
        total -= 10
        ace_count -= 1
    return total, ace_count


def final_dist(threshold):

    @lru_cache(maxsize=None)
    def dist(total, ace_count):
        if total > 21:
            return {'bust': 1.0}
        if total >= threshold:
            return {total: 1.0}
        out = defaultdict(float)
        for card, p in CARD_PROBS.items():
            t, s = add_card(total, ace_count, card)
            for outcome, q in dist(t, s).items():
                out[outcome] += p * q
        return dict(out)

    result = defaultdict(float)
    for c1, p1 in CARD_PROBS.items():
        for c2, p2 in CARD_PROBS.items():
            t, s = add_card(0, 0, c1)
            t, s = add_card(t, s, c2)
            if t == 21:
                result['blackjack'] += p1 * p2
            else:
                for outcome, q in dist(t, s).items():
                    result[outcome] += p1 * p2 * q
    return dict(result)

#table
outcomes = [15, 16, 17, 18, 19, 20, 21, 'blackjack', 'bust']
table1 = pd.DataFrame(
    {f'Stand {t}': [round(final_dist(t).get(o, 0) * 100, 2) for o in outcomes]
     for t in range(15, 21)},
    index=['15', '16', '17', '18', '19', '20', '21', 'Blackjack', 'Bust']
)
table1 = table1.replace(0.0, '–')
print(table1)


           Stand 15 Stand 16 Stand 17 Stand 18 Stand 19 Stand 20
15            13.29        –        –        –        –        –
16            12.80    13.88        –        –        –        –
17            12.27    13.36    14.51        –        –        –
18            11.71     12.8    13.95    15.18        –        –
19            11.11    12.19    13.35    14.57    15.88        –
20            15.78    16.87    18.03    19.25    20.56    21.96
21             5.03     6.12     7.27      8.5     9.81     11.2
Blackjack      4.73     4.73     4.73     4.73     4.73     4.73
Bust          13.28    20.04    28.16    37.77    49.02    62.11


In [17]:
#monte carlo simulation for part 1
def simulate(threshold=17, N=1000000):
    counts = Counter()
    for _ in range(N):
        cards, total = dealer_hand(threshold)
        if total > 21:
            counts['bust'] += 1
        elif total == 21 and len(cards) == 2:
            counts['blackjack'] += 1
        else:
            counts[total] += 1
    return {k: round(v / N, 4) for k, v in sorted(counts.items(), key=str)}


outcomes = [15, 16, 17, 18, 19, 20, 21, 'blackjack', 'bust']
labels   = ['15', '16', '17', '18', '19', '20', '21', 'Blackjack', 'Bust']

sim_results = {t: simulate(t) for t in range(15, 21)}

sim_table = pd.DataFrame(
    {f'Stand {t}': [round(sim_results[t].get(o, 0) * 100, 2) for o in outcomes]
     for t in range(15, 21)},
    index=labels
).replace(0.0, '–')

print(sim_table)

           Stand 15 Stand 16 Stand 17 Stand 18 Stand 19 Stand 20
15            13.28        –        –        –        –        –
16            12.77    13.89        –        –        –        –
17            12.31    13.27    14.47        –        –        –
18            11.75    12.79    13.95    15.17        –        –
19            11.10    12.19    13.31    14.57    15.89        –
20            15.70    16.92    18.02    19.24    20.56    22.02
21             5.04     6.13     7.29     8.49     9.82    11.17
Blackjack      4.74     4.76     4.76     4.74     4.74     4.73
Bust          13.31    20.05     28.2     37.8    48.98    62.08


In [21]:
#part 2
#house edge when the player copies the dealer (exact)

def payoff(player, dealer):
    if player == 'blackjack' and dealer == 'blackjack': return 0
    if player == 'blackjack': return 1.5
    if dealer == 'blackjack': return -1
    if player == 'bust':      return -1   # player busts first, loses regardless
    if dealer == 'bust':      return 1
    return (player > dealer) - (player < dealer)   # +1 win, -1 loss, 0 push

def house_edge(player_threshold, dealer_threshold=17):
    P = final_dist(player_threshold)
    D = final_dist(dealer_threshold)
    return -sum(pp * pd * payoff(p, d)
                for p, pp in P.items()
                for d, pd in D.items())

thresholds = range(15, 21)
table2 = pd.DataFrame(
    {'House edge (%)': [round(house_edge(t, t) * 100, 2) for t in thresholds]},
    index=[f'Both stand on {t}' for t in thresholds]
)

print(table2)

                  House edge (%)
Both stand on 15           -0.49
Both stand on 16            1.76
Both stand on 17            5.67
Both stand on 18           12.01
Both stand on 19           21.78
Both stand on 20           36.32


In [22]:
#monte carlo simulation for part 2

def player_hand(threshold=17):
    cards = [draw(), draw()]
    while hand_value(cards) < threshold:
        cards.append(draw())
    return cards, hand_value(cards)

#assume player has the strategy to copy the dealer
def symmetry_strategy_simulation(threshold = 17, N = 1000000):
  dealer_win = 0
  for _ in range(N):
    dealer_cards, dealer_total = dealer_hand(threshold)
    player_cards, player_total = player_hand(threshold)

    player_blackjack = is_blackjack(player_cards)
    dealer_blackjack = is_blackjack(dealer_cards)

    #both players get blackjack
    if player_blackjack and dealer_blackjack:
      pass
    #player gets blackjack
    elif player_blackjack:
      dealer_win -= 3/2
    #dealer gets blackjack
    elif dealer_blackjack:
      dealer_win += 1
    #player goes bust
    elif player_total > 21:
      dealer_win += 1
    #dealer goes bust
    elif dealer_total > 21:
      dealer_win -= 1
    #player gets higher numbers than dealer
    elif player_total > dealer_total:
      dealer_win -= 1
    #dealer gets higher numbers than the player
    elif player_total < dealer_total:
      dealer_win += 1

  return dealer_win/N

thresholds = range(15, 21)
sim_table2 = pd.DataFrame(
    {'House edge (%)': [round(symmetry_strategy_simulation(t) * 100, 2) for t in thresholds]},
    index=[f'Both stand on {t}' for t in thresholds]
)

print(sim_table2)

                  House edge (%)
Both stand on 15           -0.54
Both stand on 16            1.67
Both stand on 17            5.61
Both stand on 18           12.05
Both stand on 19           21.82
Both stand on 20           36.40


In [23]:
#part 3
#best player stopping threshold for each dealer stopping threshold (exact)

dealer_thresholds = range(15, 21)
player_thresholds = range(12, 21)

table3 = pd.DataFrame(
    {f'Dealer stands on {d}': [round(house_edge(p, d) * 100, 2) for p in player_thresholds]
     for d in dealer_thresholds},
    index=[f'Player stands on {p}' for p in player_thresholds]
)
print(table3)

# player's best response to each dealer rule, and the house edge that leaves
best_response = pd.DataFrame({
    'Best player threshold': [table3[col].idxmin() for col in table3.columns],
    'House edge (%)':        [table3[col].min() for col in table3.columns],
}, index=table3.columns)
print(best_response)

                     Dealer stands on 15  Dealer stands on 16  \
Player stands on 12                13.70                12.81   
Player stands on 13                 9.16                 9.91   
Player stands on 14                 4.34                 6.99   
Player stands on 15                -0.49                 4.27   
Player stands on 16                -3.46                 1.76   
Player stands on 17                -2.31                 1.57   
Player stands on 18                 3.50                 6.17   
Player stands on 19                14.57                16.21   
Player stands on 20                31.56                32.40   

                     Dealer stands on 17  Dealer stands on 18  \
Player stands on 12                 8.08                -0.93   
Player stands on 13                 6.82                -0.57   
Player stands on 14                 5.82                 0.36   
Player stands on 15                 5.28                 2.01   
Player stands on 16     

In [24]:
#optimal player strategy for each dealer stopping threshold
#part 3

#assume player has the strategy to have the ideal stopping threshold.
def stopping_threshold_strategy_simulation(dealer_threshold = 17, player_threshold = 17, N = 1000000):
  dealer_win = 0
  for _ in range(N):
    dealer_cards, dealer_total = dealer_hand(dealer_threshold)
    player_cards, player_total = player_hand(player_threshold)

    player_blackjack = is_blackjack(player_cards)
    dealer_blackjack = is_blackjack(dealer_cards)

    #both players get blackjack
    if player_blackjack and dealer_blackjack:
      pass
    #player gets blackjack
    elif player_blackjack:
      dealer_win -= 3/2
    #dealer gets blackjack
    elif dealer_blackjack:
      dealer_win += 1
    #player goes bust
    elif player_total > 21:
      dealer_win += 1
    #dealer goes bust
    elif dealer_total > 21:
      dealer_win -= 1
    #player gets higher numbers than dealer
    elif player_total > dealer_total:
      dealer_win -= 1
    #dealer gets higher numbers than the player
    elif player_total < dealer_total:
      dealer_win += 1

  return dealer_win/N

dealer_thresholds = range(15, 21)
player_thresholds = range(12, 21)

sim_table3 = pd.DataFrame(
    {f'Dealer stands on {d}': [round(stopping_threshold_strategy_simulation(d, p) * 100, 2)
                               for p in player_thresholds]
     for d in dealer_thresholds},
    index=[f'Player stands on {p}' for p in player_thresholds]
)
print(sim_table3)

sim_best_response = pd.DataFrame({
    'Best player threshold': [sim_table3[col].idxmin() for col in sim_table3.columns],
    'House edge (%)':        [sim_table3[col].min() for col in sim_table3.columns],
}, index=sim_table3.columns)
print(sim_best_response)

                     Dealer stands on 15  Dealer stands on 16  \
Player stands on 12                13.76                12.87   
Player stands on 13                 9.22                 9.92   
Player stands on 14                 4.59                 6.91   
Player stands on 15                -0.56                 4.22   
Player stands on 16                -3.36                 1.74   
Player stands on 17                -2.10                 1.56   
Player stands on 18                 3.62                 6.38   
Player stands on 19                14.52                16.29   
Player stands on 20                31.60                32.46   

                     Dealer stands on 17  Dealer stands on 18  \
Player stands on 12                 7.93                -0.96   
Player stands on 13                 6.80                -0.51   
Player stands on 14                 5.83                 0.35   
Player stands on 15                 5.09                 2.02   
Player stands on 16     